In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns
from epiweeks import Week
import matplotlib.pyplot as plt

In [2]:
state = 'AM'
df_climate = pd.read_parquet(f'../climate/{state}_climate.parquet', columns = ['geocodigo', 'temp_med', 'umid_med', 'precip_tot'])

df_climate.head()

,geocodigo,temp_med,umid_med,precip_tot
date,,,,
2017-09-02,1300102,27.488450,81.164970,0.06182
2017-09-03,1300102,28.568272,79.854485,0.49388
2017-09-04,1300102,28.503593,83.128330,0.22797
2017-09-05,1300102,27.543789,82.803660,0.11151
2017-09-06,1300102,27.388054,82.039130,0.03625


In [3]:
df_climate.loc[df_climate.precip_tot > 10].shape[0]/df_climate.shape[0]

0.021808564927384835

In [4]:
df_climate.precip_tot.max()

114.40864

In [5]:
def add_epiweek_label(df_w):
    '''
    This function assumes that the dataframe has a datetime index
    and add the epiweek and year value
    '''

    df_w['epiweek_label'] = [Week.fromdate(x) for x in df_w.index]

    df_w['epiweek_label'] = df_w['epiweek_label'].astype(str)

    df_w['epiweek'] = df_w['epiweek_label'].astype(str).str[-2:].astype(int)
    
    df_w['year'] = df_w['epiweek_label'].astype(str).str[:4].astype(int)

    return df_w


def load_agg_clima(state, ini_date = '2010-01-01', end_date ='2020-12-31'):
    '''
    This function load and aggregates the climatic variables by the epidemiological year
    '''
    df_climate = pd.read_parquet(f'../climate/{state}_climate.parquet', columns = ['geocodigo', 'temp_med', 'umid_med', 'precip_tot'])
    
    df_climate.index = pd.to_datetime(df_climate.index)
        
    df_climate = df_climate.loc[(df_climate.index >= ini_date) & (df_climate.index<= end_date)].sort_index()

    df_climate['thr_temp'] = 0

    df_climate.loc[df_climate.temp_med > 20, 'thr_temp'] = 1

    df_climate['thr_umid'] = 0

    df_climate.loc[df_climate.umid_med > 60, 'thr_umid'] = 1

    df_climate['thr_prec'] = 0

    df_climate.loc[df_climate.precip_tot > 10, 'thr_prec'] = 1
    
    df_climate = add_epiweek_label(df_climate)

    df_climate['thr_temp_umid'] = 0

    df_climate.loc[(df_climate.temp_med > 20) & (df_climate.umid_med > 60), 'thr_temp_umid'] = 1
    
    df_clima_agg = pd.DataFrame()
    
    for year in df_climate.year.unique():
    
        filter1 = (df_climate.year == year) & (df_climate.epiweek >= 41)
        filter2 = (df_climate.year == year+1) & (df_climate.epiweek < 41)
        
        df_ = df_climate.loc[filter1 | filter2].reset_index().groupby('geocodigo').agg({'temp_med': 'mean', 'umid_med': 'mean', 'precip_tot': 'sum',
                                                                                       'thr_temp':'sum',
                                                                                       'thr_umid':'sum', 
                                                                                       'thr_prec':'sum',
                                                                                       'thr_temp_umid': 'sum'}).reset_index()
    
        df_['year'] = year
        
        df_clima_agg = pd.concat([df_clima_agg, df_])
    
    return df_clima_agg    

In [6]:
states_BR = ['AL',
 'BA',
 'CE',
 'MA',
 'PB',
 'PE',
 'PI',
 'SE',
 'RN',
 'SP',
 'MG',
 'RJ',
 'ES',
 'AM',
 'AP',
 'TO',
 'RR',
 'RO',
 'AC',
 'PA',
 'DF',
 'GO',
 'MT',
 'MS',
 'RS',
 'SC',
 'PR']


### Clima no período 2010_2019: 

In [7]:
state = 'SC'
df_ = load_agg_clima(state)
df_ = df_.loc[df_.year.isin(np.arange(2010, 2020))]
df_.year.unique()

array([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019])

In [8]:
df_

,geocodigo,temp_med,umid_med,precip_tot,thr_temp,thr_umid,thr_prec,thr_temp_umid,year
0,4200051,16.872875,81.178131,691.745629,103,363,11,103,2010
1,4200101,18.156404,78.375269,784.155021,154,350,13,150,2010
2,4200200,16.072065,85.040855,779.850467,79,361,16,79,2010
3,4200309,16.908791,85.087599,838.112567,102,361,15,102,2010
4,4200408,17.133007,80.196167,724.789073,111,361,11,111,2010
...,...,...,...,...,...,...,...,...,...
290,4219507,20.088224,70.012126,431.398802,208,291,9,144,2019
291,4219606,19.292917,72.481074,458.823403,187,322,9,151,2019
292,4219705,19.292917,72.481074,458.823403,187,322,9,151,2019
293,4219853,18.266463,72.970675,449.097790,145,331,7,120,2019


In [9]:
df_perfis = pd.read_csv('../Determinantes/Dengue/dengue_pattern_10_19.csv', sep = ';', usecols = ['muni_code', 'dengue_pattern'])

df_perfis.head()

,muni_code,dengue_pattern
0,1100015,Episódico/Epidêmico
1,1100023,Epidêmico
2,1100031,Episódico/Epidêmico
3,1100049,Epidêmico
4,1100056,Episódico/Epidêmico


The cell below concatenate the aggregated dataframes of all states in just one:

In [10]:
df_climate_all = pd.DataFrame()

for state in states_BR: 
    
    df_climate = load_agg_clima(state)

    df_climate = df_climate.loc[df_climate.year.isin(np.arange(2010, 2020))]
    
    df_climate_all = pd.concat([df_climate_all, df_climate])

# merge the climatic dataset with the dengue patterns 
df_climate_all = df_climate_all.merge(df_perfis, left_on = 'geocodigo', right_on = 'muni_code').drop('muni_code', axis =1)
    
df_climate_all.head()

,geocodigo,temp_med,umid_med,precip_tot,thr_temp,thr_umid,thr_prec,thr_temp_umid,year,dengue_pattern
0,2700102,25.917411,71.160050,174.737289,364,314,2,314,2010,Episódico/Epidêmico
1,2700201,25.484200,80.407899,424.205669,364,364,5,364,2010,Episódico/Epidêmico
2,2700300,25.588142,77.216115,277.945870,364,364,2,364,2010,Epidêmico
3,2700409,24.897218,81.989589,423.138619,364,364,4,364,2010,Episódico/Epidêmico
4,2700508,25.800147,78.780638,491.421552,364,363,5,363,2010,Episódico


Verifying the presence of null values in the data: 

In [11]:
df_climate_all.isnull().sum()

geocodigo         0
temp_med          0
umid_med          0
precip_tot        0
thr_temp          0
thr_umid          0
thr_prec          0
thr_temp_umid     0
year              0
dengue_pattern    0
dtype: int64

Save the dataframe: 

In [12]:
%%time
df_climate_all.to_csv('../Determinantes/clima/clima_agg_10_19.csv')

CPU times: user 220 ms, sys: 16.3 ms, total: 236 ms
Wall time: 293 ms


### Clima no período 2020_2022:

In [13]:
state = 'AC'
df_ = load_agg_clima(state, ini_date = '2020-01-01', end_date = '2023-12-31')
df_ = df_.loc[df_.year.isin(np.arange(2020, 2023))]
df_.year.unique()

array([2020, 2021, 2022])

In [14]:
df_perfis = pd.read_csv('../Determinantes/Dengue/dengue_pattern_20_22.csv', sep = ';', usecols = ['muni_code', 'dengue_pattern'])

df_perfis.head()

,muni_code,dengue_pattern
0,1100015,Epidêmico
1,1100023,Epidêmico
2,1100031,Episódico/Epidêmico
3,1100049,Epidêmico
4,1100056,Episódico/Epidêmico


In [15]:
df_climate_all = pd.DataFrame()

for state in states_BR: 
    
    df_climate = load_agg_clima(state, ini_date = '2020-01-01', end_date = '2023-12-31')

    df_climate = df_climate.loc[df_climate.year.isin(np.arange(2020, 2023))]
    
    df_climate_all = pd.concat([df_climate_all, df_climate])

# merge the climatic dataset with the dengue patterns 
df_climate_all = df_climate_all.merge(df_perfis, left_on = 'geocodigo', right_on = 'muni_code').drop('muni_code', axis = 1)
    
df_climate_all.head()

,geocodigo,temp_med,umid_med,precip_tot,thr_temp,thr_umid,thr_prec,thr_temp_umid,year,dengue_pattern
0,2700102,26.368572,65.759019,79.313571,371,241,0,241,2020,Episódico/Epidêmico
1,2700201,25.656351,78.530554,320.167782,371,371,3,371,2020,Episódico/Epidêmico
2,2700300,25.839300,73.983355,187.897610,371,366,1,366,2020,Epidêmico
3,2700409,25.007031,80.555151,336.263240,371,371,3,371,2020,Episódico/Epidêmico
4,2700508,25.844227,79.239810,418.317611,371,371,4,371,2020,Episódico


In [16]:
df_climate_all.isnull().sum()

geocodigo         0
temp_med          0
umid_med          0
precip_tot        0
thr_temp          0
thr_umid          0
thr_prec          0
thr_temp_umid     0
year              0
dengue_pattern    0
dtype: int64

In [17]:
df_climate_all.to_csv('../Determinantes/clima/clima_agg_20_22.csv')